# Provisional Customer Perk Assignments

This notebook assigns one proposed invitation benefit to every customer using explicit pilot rules. It preserves candidate RFM groups as a separate activity/value dimension. It does not infer verified favorite perks, establish causal effectiveness, or finalize unified marketing segments.

**Inputs:** `data/own_processed/user_features_windowed.csv` and the group-assignment output of notebook 04.

**Outputs:** `reports/provisional_perk_assignments/`. Source files and earlier reports are not modified. Repeated runs replace only this notebook's derived outputs.

The rule order and thresholds are analyst-designed assumptions. Every record includes a rationale, qualitative evidence label, fallback flag and overlapping eligibility. Two alternate threshold sets provide a limited sensitivity check. Unsupported preferences remain explicitly unverified.

Dependencies: pandas.


In [ ]:
from pathlib import Path
import pandas as pd

print("STEP 1: Load the study feature table and candidate groups", flush=True)
PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "data" / "own_processed" / "user_features_windowed.csv").is_file()), None)
if PROJECT is None:
    raise FileNotFoundError("Expected data/own_processed/user_features_windowed.csv. No output written.")
SOURCE = PROJECT / "data" / "own_processed" / "user_features_windowed.csv"
GROUPS = PROJECT / "reports" / "rfm_lrfm_candidate_profiles" / "all_customers_candidate_assignments.csv"
if not GROUPS.is_file():
    raise FileNotFoundError("Run notebook 04 first. No output written.")
OUT = PROJECT / "reports" / "provisional_perk_assignments"
d = pd.read_csv(SOURCE, dtype={"user_id": "string"})
g = pd.read_csv(GROUPS, usecols=["user_id", "candidate_group"], dtype={"user_id": "string"})
assert d.user_id.is_unique and g.user_id.is_unique and set(d.user_id) == set(g.user_id)
d = d.merge(g, on="user_id", how="left", validate="one_to_one")

# Analyst-designed pilot rules, not learned preference or treatment-response models.
# Priority favors observed flexibility problems and concrete baggage use over broad applicability.
# Alternative priorities may also be reasonable; thresholds and ordering require testing.
CONFIGS = {
    "baseline": {"cancel_rate": 0.25, "bags": 0.5, "discount_rate": 0.25},
    "lower_thresholds": {"cancel_rate": 0.20, "bags": 0.25, "discount_rate": 0.20},
    "higher_thresholds": {"cancel_rate": 0.50, "bags": 1.0, "discount_rate": 0.50},
}
PERKS = ["Free hotel meal", "Free checked bag", "No cancellation fees", "Exclusive discounts", "One free hotel night with a flight"]

def assign(frame, config):
    f = frame
    only_cancel = f.booking_status.eq("Only canceled bookings")
    repeated_cancellation = f.num_original_booked_trips.ge(2) & f.cancellation_rate.ge(config["cancel_rate"]) & f.num_canceled_trips.gt(0)
    bag = f.num_flight_trips.ge(2) & f.average_checked_bags.ge(config["bags"])
    disc_f = f.num_flight_trips.ge(2) & f.flight_discounted_booking_count.ge(2) & f.flight_discounted_booking_rate.ge(config["discount_rate"])
    disc_h = f.num_hotel_trips.ge(2) & f.hotel_discounted_booking_count.ge(2) & f.hotel_discounted_booking_rate.ge(config["discount_rate"])
    bundle = f.flight_hotel_combined_trip_count.ge(2)
    hotel = f.num_hotel_trips.ge(1)
    gates = [("No cancellation fees", only_cancel | repeated_cancellation,
              "Observed cancellation history; fee exposure and flexibility preference are unknown.", "Low: flexibility hypothesis"),
             ("Free checked bag", bag,
              "At least two noncanceled flight bookings and mean checked bags meets the rule threshold.", "Moderate: repeated use; preference unverified"),
             ("Exclusive discounts", disc_f | disc_h,
              "Repeated discounted purchases and product-specific discount rate meets the rule threshold; no causal price sensitivity inferred.", "Low: purchase association"),
             ("One free hotel night with a flight", bundle,
              "At least two observed combined flight–hotel trips; benefit is applicable, not proven preferred.", "Low: bundle applicability"),
             ("Free hotel meal", hotel,
              "Observed hotel booking supports applicability only; meal purchases and meal preferences are unavailable.", "Low: hotel applicability" )]
    result = f[["user_id", "candidate_group", "baseline_status", "booking_status"]].copy()
    result["proposed_perk"] = "Exclusive discounts"
    result["assignment_reason"] = "Generic invitation-benefit pilot fallback; insufficient rule evidence for a personalized preference."
    result["evidence_confidence"] = "Low: generic fallback"
    result["assignment_is_fallback"] = True
    result["eligible_rule_count"] = sum(mask.astype(int) for _, mask, _, _ in gates)
    result["eligible_perks"] = ["; ".join(perk for perk, mask, _, _ in gates if bool(mask.iloc[i])) for i in range(len(f))]
    unresolved = pd.Series(True, index=f.index)
    for perk, mask, reason, confidence in gates:
        use = unresolved & mask
        result.loc[use, "proposed_perk"] = perk
        result.loc[use, "assignment_reason"] = reason
        result.loc[use, "evidence_confidence"] = confidence
        result.loc[use, "assignment_is_fallback"] = False
        unresolved &= ~mask
    result["priority_resolved_overlap"] = result.eligible_rule_count.gt(1)
    result["preference_verified"] = False
    return result

print("STEP 2: Apply provisional rules and sensitivity checks", flush=True)
a = assign(d, CONFIGS["baseline"])
assert len(a) == len(d) and a.user_id.is_unique and a.proposed_perk.notna().all()
assert set(a.proposed_perk).issubset(PERKS)
sensitivity = []
for name, config in CONFIGS.items():
    alt = assign(d, config)
    changed = int(alt.proposed_perk.ne(a.proposed_perk).sum())
    for perk in PERKS:
        count = int(alt.proposed_perk.eq(perk).sum())
        sensitivity.append({"rule_set": name, "perk": perk, "customers": count,
                            "share_of_full_cohort": count / len(d),
                            "customers_changed_vs_baseline": changed,
                            "share_changed_vs_baseline": changed / len(d)})
s = pd.DataFrame(sensitivity)
assert s.groupby("rule_set").customers.sum().eq(len(d)).all()
coverage = (a.groupby(["candidate_group", "proposed_perk", "evidence_confidence", "assignment_is_fallback"], dropna=False)
            .size().reset_index(name="customers"))
assert coverage.customers.sum() == len(d)

print("STEP 3: Save assignments and the rule documentation", flush=True)
OUT.mkdir(parents=True, exist_ok=True)
a.to_csv(OUT / "customer_perk_assignments_provisional.csv", index=False)
coverage.to_csv(OUT / "perk_assignment_coverage.csv", index=False)
s.to_csv(OUT / "perk_rule_sensitivity.csv", index=False)
counts = a.proposed_perk.value_counts().reindex(PERKS, fill_value=0)
lines = ["# Provisional Perk Assignment Review", "",
         f"Customers retained and assigned: {len(a):,}. These are pilot recommendations, not verified favorite perks.",
         "", "## Baseline assignment counts"]
lines += [f"- {perk}: {int(count):,}" for perk,count in counts.items()]
lines += [f"- Generic fallback assignments: {int(a.assignment_is_fallback.sum()):,}.",
          f"- Customers with multiple eligible rules: {int(a.priority_resolved_overlap.sum()):,}.",
          "", "## Rules and priority", "1. No cancellation fees: only-canceled status, or at least two original booked trips with a cancellation and cancellation rate at least 25%.",
          "2. Free checked bag: at least two noncanceled flight bookings and average checked bags at least 0.5 per flight booking.",
          "3. Exclusive discounts: at least two discounted bookings for the same product, with that product's discounted-booking rate at least 25%.",
          "4. One free hotel night with a flight: at least two observed combined flight–hotel trips.",
          "5. Free hotel meal: at least one observed noncanceled hotel booking; this is broad applicability only.",
          "6. Otherwise: exclusive discounts as a generic invitation pilot fallback, explicitly low-confidence—not evidence of price sensitivity.",
          "", "## Evidence and uncertainty", "Thresholds and precedence are analyst-designed assumptions, not estimated optimal rules. Cancellation fees, meal preferences, benefit response and operational benefit costs are unavailable. No recommendation has verified preference; confidence labels describe qualitative behavioral support, not probabilities. Average bags is not a baggage-use rate and is not adjusted for party size. Behavioral evidence uses exported history, while RFM Frequency and Monetary use the fixed recent window. Opportunity to exhibit behavior differs by customer. Unknown spending does not prevent behavior-based recommendations and is never imputed to zero. Cohort eligibility already selects recent website engagement; no-booking fallback does not prove dormancy or churn.",
          "", "## Sensitivity", "Lower thresholds: cancellation 20%, mean bags 0.25, product discount rate 20%. Higher thresholds: cancellation 50%, mean bags 1, product discount rate 50%. Repeated-booking gates and priority are held fixed; this check does not establish stability to changing rule order or window."]
for name in ["lower_thresholds", "higher_thresholds"]:
    item = s.loc[s.rule_set.eq(name)].iloc[0]
    lines.append(f"- {name}: {int(item.customers_changed_vs_baseline):,} customers ({item.share_changed_vs_baseline:.1%}) change proposed perk.")
lines += ["", "## Segment structure", "Candidate RFM activity/value groups and proposed perks are separate dimensions. Outside-comparison statuses remain explicit handling strata, not automatically meaningful marketing segments. This output is not a final unified segmentation. All customers must be accounted for in the final deliverable; presentation highlights can focus on approximately three well-supported groups.",
          "", "## Validation plan", "Compare the selected perk message against a generic message or alternative perk using randomized customer-level assignment. Measure sign-up and incremental booking outcomes; measure contribution margin only if benefit costs and margins become available. Analyze all randomized customers, not only email openers. Preference elicitation can help validate weak assignments. Do not present rule counts as evidence of effectiveness.",
          "", "## Next review", "Review concentration, fallback coverage, overlapping eligibility and threshold sensitivity before approving the rules or creating final presentation highlights."]
(OUT / "perk_assignment_summary.md").write_text("\n".join(lines), encoding="utf-8")
print("COMPLETE:", OUT, flush=True)
print("Source feature table and earlier reports were not modified.", flush=True)
